# SHAP Feature Importance & Stability Analysis
**Project**: Horizon- and Regime-Specific ML for Stock Return Prediction  
**Market**: Indian Equity (NSE) exclusively  
**Run ID**: shap_explainability_models  

## What this notebook does
1. Re-trains XGBoost (same frozen hyperparameters from models.yaml) on each of 6 walk-forward splits × 2 horizons
2. Computes SHAP TreeExplainer values on each test set
3. Reports:
   - Global feature importance (mean |SHAP|) per split × horizon
   - Per-regime SHAP importance (Low / Medium / High vol)
   - Cross-split feature ranking stability (Spearman correlation, Kendall's W)
4. Answers **RQ6**: Are important features stable across regimes and retraining windows?

## Leakage safeguards
- XGBoost re-trained with exact same hyperparams as tabular ML Stage 6 (no new tuning)
- Scaler fit on training data only before SHAP computation
- SHAP values computed on TEST data only (no training data leakage)
- Regime labels assigned from training-window thresholds (same as tabular ML Stage 6)


In [ ]:
# --- CELL 1: AUTO-DETECT PATHS ----------------------------------------
import os

def find_project_dirs(input_root='/kaggle/input'):
    config_dir = data_dir = None
    for root, dirs, files in os.walk(input_root):
        depth = root.replace(input_root, '').count(os.sep)
        if depth > 5: dirs[:] = []; continue
        if config_dir is None and 'configs' in dirs:
            candidate = os.path.join(root, 'configs')
            if os.path.isfile(os.path.join(candidate, 'experiment.yaml')):
                config_dir = candidate
        if data_dir is None and 'processed' in dirs:
            candidate = os.path.join(root, 'processed')
            if os.path.isfile(os.path.join(candidate, 'features_targets.csv')):
                data_dir = candidate
        if config_dir and data_dir: break
    return config_dir, data_dir

CONFIG_DIR, DATA_DIR = find_project_dirs()
if CONFIG_DIR is None or DATA_DIR is None:
    print('ERROR: dataset not found. Checking structure...')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input','').count(os.sep)
        if depth>4: dirs[:] = []; continue
        print('  '*depth + os.path.basename(root) + '/')
        for f in files[:3]: print('  '*depth + '  ' + f)
    raise RuntimeError('Dataset not found.')

OUTPUT_DIR  = '/kaggle/working/shap_explainability_output'
RUN_ID      = 'shap_explainability_models'
RANDOM_SEED = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'CONFIG_DIR: {CONFIG_DIR}')
print(f'DATA_DIR  : {DATA_DIR}')


In [ ]:
# --- CELL 2: INSTALL & IMPORTS ------------------------------------------
!pip install shap xgboost scikit-learn pyyaml pandas numpy --quiet

import warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yaml
import shap
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from scipy import stats

np.random.seed(RANDOM_SEED)
print(f'shap={shap.__version__}')


In [ ]:
# --- CELL 3: LOAD DATA & CONFIG -----------------------------------------
with open(f'{CONFIG_DIR}/experiment.yaml') as f:
    exp_cfg = yaml.safe_load(f)
with open(f'{CONFIG_DIR}/models.yaml') as f:
    mdl_cfg = yaml.safe_load(f)

SPLITS_CFG = exp_cfg['walk_forward']['splits']
HORIZONS   = exp_cfg['horizons']
BENCHMARK  = exp_cfg['data']['benchmark']
XGB_PARAMS = mdl_cfg['xgboost']

df = pd.read_csv(f'{DATA_DIR}/features_targets.csv', parse_dates=['Date'])
with open(f'{DATA_DIR}/feature_list.txt') as f:
    FEATURE_COLS = [l.strip() for l in f if l.strip()]

df_stocks = df[df['Ticker'] != BENCHMARK].copy().reset_index(drop=True)
REGIME_NAMES = {0: 'Low', 1: 'Medium', 2: 'High'}

print(f'Data: {df_stocks.shape}  Features: {len(FEATURE_COLS)}')
print(f'Splits: {len(SPLITS_CFG)}  Horizons: {HORIZONS}')
print(f'XGB params: {XGB_PARAMS}')


In [ ]:
# --- CELL 4: HELPERS ----------------------------------------------------
ALL_DATES = np.sort(df_stocks['Date'].values.astype('datetime64[ns]'))
EMBARGO_DAYS = 5

def get_purge_cutoff(test_start, horizon):
    ts_np  = np.datetime64(pd.Timestamp(test_start), 'ns')
    ts_idx = np.searchsorted(ALL_DATES, ts_np, side='left')
    return pd.Timestamp(ALL_DATES[max(ts_idx - horizon, 0)])

def get_embargo_cutoff(purge_cutoff, n):
    pc_np  = np.datetime64(purge_cutoff, 'ns')
    pc_idx = np.searchsorted(ALL_DATES, pc_np, side='right')
    return pd.Timestamp(ALL_DATES[min(pc_idx + n - 1, len(ALL_DATES)-1)])

def clean(X):
    return np.where(np.isfinite(X), X, 0.0)

def spearman_r(a, b):
    return stats.spearmanr(a, b).correlation

def kendalls_w(rank_matrix):
    """rank_matrix: shape (n_raters, n_items)"""
    m, n = rank_matrix.shape
    S = np.sum((rank_matrix.sum(axis=0) - m*(n+1)/2)**2)
    W = 12*S / (m**2 * (n**3 - n))
    chi2 = m*(n-1)*W
    p    = 1 - stats.chi2.cdf(chi2, df=n-1)
    return round(W, 4), round(chi2, 4), round(p, 4)

print('Helpers ready.')


In [ ]:
# --- CELL 5: XGB RETRAIN + SHAP PER SPLIT × HORIZON --------------------
# NOTE: Re-training uses IDENTICAL hyperparams from models.yaml.
# This is NOT new hyperparameter tuning — it is reproducing the tabular ML Stage 6
# model to obtain model objects needed for SHAP (not saved in tabular ML Stage 6).

all_shap_rows  = []   # per test-sample SHAP values (mean |SHAP|)
imp_by_split   = []   # global importance per split × horizon
regime_imp     = []   # importance per split × horizon × regime

# Clean XGB params (remove keys sklearn doesn't accept)
xgb_params = {k: v for k, v in XGB_PARAMS.items()
               if k not in ('use_label_encoder', 'eval_metric')}
xgb_params['random_state'] = RANDOM_SEED
xgb_params['n_jobs']       = -1

total = len(SPLITS_CFG) * len(HORIZONS)
combo = 0

for split_cfg in SPLITS_CFG:
    label      = split_cfg['label']
    train_start = pd.Timestamp(split_cfg['train_start'])
    test_start  = pd.Timestamp(split_cfg['test_start'])
    test_end    = pd.Timestamp(split_cfg['test_end'])
    regime_col  = f'regime_{label}'

    for horizon in HORIZONS:
        combo += 1
        t0 = time.time()
        target_col = f'fwd_label_{horizon}d'
        print(f'[{combo}/{total}] {label} h={horizon}d', flush=True)

        # Purge + embargo
        purge_cutoff = get_purge_cutoff(test_start, horizon)

        train_mask = (
            (df_stocks['Date'] >= train_start) &
            (df_stocks['Date'] <= purge_cutoff) &
            df_stocks[target_col].notna()
        )
        test_mask = (
            (df_stocks['Date'] >= test_start) &
            (df_stocks['Date'] <= test_end) &
            df_stocks[target_col].notna()
        )

        df_train = df_stocks[train_mask].copy()
        df_test  = df_stocks[test_mask].copy()

        if len(df_train) < 100 or len(df_test) < 10:
            print(f'  SKIP: insufficient data')
            continue

        # Scale on training only
        X_train = clean(df_train[FEATURE_COLS].values)
        X_test  = clean(df_test[FEATURE_COLS].values)
        y_train = df_train[target_col].values
        y_test  = df_test[target_col].values

        scaler  = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc  = scaler.transform(X_test)

        # Train XGBoost (identical to tabular ML Stage 6)
        model = XGBClassifier(**xgb_params)
        model.fit(X_train_sc, y_train, verbose=False)

        # SHAP TreeExplainer on TEST data
        explainer    = shap.TreeExplainer(model)
        shap_values  = explainer.shap_values(X_test_sc)  # shape: (n_test, n_features)
        mean_abs_shap = np.abs(shap_values).mean(axis=0) # (n_features,)

        # Global importance row
        imp_row = {'split': label, 'horizon': horizon}
        imp_row.update(dict(zip(FEATURE_COLS, mean_abs_shap)))
        imp_by_split.append(imp_row)

        # Per-regime importance
        if regime_col in df_test.columns:
            for rid, rname in REGIME_NAMES.items():
                rmask = df_test[regime_col].values == rid
                if rmask.sum() < 5: continue
                regime_shap = np.abs(shap_values[rmask]).mean(axis=0)
                reg_row = {'split': label, 'horizon': horizon, 'regime': rname,
                           'n': int(rmask.sum())}
                reg_row.update(dict(zip(FEATURE_COLS, regime_shap)))
                regime_imp.append(reg_row)

        # Top-10 features for this split
        top10_idx = mean_abs_shap.argsort()[::-1][:10]
        top10 = [(FEATURE_COLS[i], round(float(mean_abs_shap[i]), 6)) for i in top10_idx]
        print(f'  Top-10: {top10[:5]}... ({time.time()-t0:.0f}s)', flush=True)

print('=== SHAP computation complete ===')


In [ ]:
# --- CELL 6: FEATURE IMPORTANCE STABILITY ANALYSIS ---------------------
imp_df = pd.DataFrame(imp_by_split)

stability_rows = []

for horizon in HORIZONS:
    sub = imp_df[imp_df['horizon'] == horizon].drop(columns=['split','horizon'])
    splits = imp_df[imp_df['horizon'] == horizon]['split'].tolist()

    # Mean importance across all splits
    mean_imp = sub.mean().sort_values(ascending=False)
    print(f'\n--- h={horizon}d: Top-15 features (mean |SHAP| across splits) ---')
    print(mean_imp.head(15).round(6).to_string())

    # Cross-split Spearman correlation of importance rankings
    imp_matrix = sub.values   # shape: (n_splits, n_features)
    n_sp = len(splits)
    corr_vals = []
    for i in range(n_sp):
        for j in range(i+1, n_sp):
            r = spearman_r(imp_matrix[i], imp_matrix[j])
            corr_vals.append(r)
            stability_rows.append({
                'horizon': horizon,
                'split_a': splits[i], 'split_b': splits[j],
                'spearman_r': round(r, 4)
            })

    mean_corr = np.mean(corr_vals)
    print(f'  Mean pairwise Spearman r: {mean_corr:.4f}')

    # Kendall's W on importance rankings
    rank_matrix = np.array([(-row).argsort().argsort() for row in imp_matrix])
    W, chi2, p = kendalls_w(rank_matrix)
    print(f'  Kendall W={W}  chi2={chi2}  p={p}')

    stability_rows.append({
        'horizon': horizon, 'split_a': 'ALL', 'split_b': 'ALL',
        'spearman_r': round(mean_corr, 4),
        'kendalls_W': W, 'chi2': chi2, 'p_value': p,
    })

stability_df = pd.DataFrame(stability_rows)

# Regime comparison: do important features change between regimes?
print('\n--- Per-regime feature rank correlation (1d horizon) ---')
reg_df = pd.DataFrame(regime_imp)
h1_reg = reg_df[reg_df['horizon'] == 1]
for reg in ['Low', 'Medium', 'High']:
    for reg2 in ['Low', 'Medium', 'High']:
        if reg >= reg2: continue
        r1 = h1_reg[h1_reg['regime']==reg][FEATURE_COLS].mean()
        r2 = h1_reg[h1_reg['regime']==reg2][FEATURE_COLS].mean()
        if len(r1) > 0 and len(r2) > 0:
            corr = spearman_r(r1.values, r2.values)
            print(f'  {reg} vs {reg2}: Spearman r = {corr:.4f}')

print('Stability analysis done.')


In [ ]:
# --- CELL 7: SAVE OUTPUTS -----------------------------------------------
imp_df     = pd.DataFrame(imp_by_split)
regime_df  = pd.DataFrame(regime_imp)
stab_df    = pd.DataFrame(stability_rows)

# Add feature ranks within each split × horizon
feat_rank_rows = []
for horizon in HORIZONS:
    sub = imp_df[imp_df['horizon']==horizon]
    mean_imp = sub[FEATURE_COLS].mean().sort_values(ascending=False)
    for rank, (feat, imp_val) in enumerate(mean_imp.items(), 1):
        feat_rank_rows.append({
            'feature': feat, 'horizon': horizon,
            'mean_abs_shap': round(float(imp_val), 6),
            'mean_rank': rank,
            # Per-split importance
            **{f'imp_{row["split"]}': round(float(row[feat]), 6)
               for _, row in sub.iterrows()}
        })
feat_rank_df = pd.DataFrame(feat_rank_rows)

imp_df.to_csv(f'{OUTPUT_DIR}/shap_importance_by_split.csv', index=False)
regime_df.to_csv(f'{OUTPUT_DIR}/shap_importance_by_regime.csv', index=False)
stab_df.to_csv(f'{OUTPUT_DIR}/shap_stability.csv', index=False)
feat_rank_df.to_csv(f'{OUTPUT_DIR}/feature_rankings.csv', index=False)

print('Saved:')
print(f'  shap_importance_by_split.csv  : {len(imp_df)} rows')
print(f'  shap_importance_by_regime.csv : {len(regime_df)} rows')
print(f'  shap_stability.csv            : {len(stab_df)} rows')
print(f'  feature_rankings.csv          : {len(feat_rank_df)} rows')

# Print final top-10 for each horizon
for h in HORIZONS:
    sub = feat_rank_df[feat_rank_df['horizon']==h].head(10)
    print(f'\nTop-10 features (h={h}d, mean |SHAP|):')
    print(sub[['feature','mean_abs_shap','mean_rank']].to_string(index=False))

# Stability summary
for h in HORIZONS:
    summary_row = stab_df[(stab_df['horizon']==h) & (stab_df['split_a']=='ALL')]
    if not summary_row.empty:
        print(f'\nh={h}d stability: Spearman r={summary_row["spearman_r"].iloc[0]:.4f}',
              f' Kendall W={summary_row.get("kendalls_W", pd.Series([None])).iloc[0]}')

print('\n=== SHAP analysis complete. Proceed to ZIP export. ===')


In [ ]:
# --- CELL 8: FINAL ZIP EXPORT -------------------------------------------
import shutil, os

zip_base = f'/kaggle/working/{RUN_ID}_artifacts'
shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
final_zip = zip_base + '.zip'
size_mb = os.path.getsize(final_zip) / 1e6
print(f'ZIP: {final_zip}  ({size_mb:.2f} MB)')
print()
print('Next steps:')
print('  1. Download ZIP from Kaggle Output tab')
print(f'  2. Extract to  artifacts/kaggle/{RUN_ID}/')
print('  3. Agent will read feature_rankings.csv + shap_stability.csv')
print('     and update RESULTS.md with RQ6 answer')
print('  4. Proceed to research reporting & synthesis')
